# Zero-Shot Irony Classification — Gemma3-4B Results



## Step 1 — Imports & Configuration

In [ ]:
import re
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score

CFG = {
    "c1_path" : "results_gemma_zeroshot_condition1.csv",
    "c2_path" : "results_gemma_zeroshot_condition2.csv",
    "c3_path" : "results_gemma_zeroshot_condition3.csv",
    "output"  : "results_gemma3_zeroshot.txt",
}

RESULTS_FILE = CFG["output"]

def log(text=''):
    print(text)
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")

# Initialise results file  ← BUG 1 FIX: was 'OLMo2'
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    f.write("Zero-Shot Irony Classification Results — Gemma3-4B\n")
    f.write(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Results will be written to: {RESULTS_FILE}")

## Step 2 — Load Data

In [ ]:
c1 = pd.read_csv(CFG["c1_path"])
c2 = pd.read_csv(CFG["c2_path"])
c3 = pd.read_csv(CFG["c3_path"])

c1 = c1.rename(columns={"context-level": "level", "basic_utterance": "base_utterance"})
c2 = c2.rename(columns={"cg_level": "level"})
c3 = c3.rename(columns={"prompt_level": "level"})

c1["condition"] = "C1_context_richness"
c2["condition"] = "C2_common_ground"
c3["condition"] = "C3_prompting_style"

COLS = ["item_id", "condition", "level", "irony_label", "base_utterance",
        "full_prompt", "model_output"]
df = pd.concat([c1[COLS], c2[COLS], c3[COLS]], ignore_index=True)
df = df.dropna(subset=["irony_label", "model_output"]).reset_index(drop=True)
df["label"] = (df["irony_label"] == "ironic").astype(int)

print(f"Total rows: {len(df)}")
print(df[["condition", "level", "irony_label"]].value_counts().sort_index().to_string())

## Step 3 — Prediction Parser (Fixed)

Gemma3 uses four distinct response styles that the original OLMo2 parser did not handle:

| Style | Example | Bug | Fix |
|---|---|---|---|
| Direct yes/no | `Yes.\n\nIt's ironic because…` | Handled correctly | No change |
| **Prefixed yes** | `Potentially yes.` | Missed by Stage 1 → keyword fallback | Added to Stage 1 |
| **Hedging** | `This could be ironic.` / `This depends on context.` | Fell to keyword count, often mislabelled | New Stage 0 → `uncertain` |
| **Chain-of-thought (C3)** | `Let's break down this statement:\n\n1. **Literal meaning…` | All truncated, no verdict | All → `uncertain`; see warning below |

In [ ]:
# ── Gemma-specific hedging patterns → always uncertain ────────────────
# BUG 3 FIX: these were previously falling to keyword counting and being
# silently mislabelled (14 C1 items, 24 C2 items).
HEDGING_PATTERNS = re.compile(
    r'^(this (could|can|may|might) be ironic'
    r'|this depends'
    r'|it (could|can|may|might) be ironic'
    r'|it depends)',
    re.IGNORECASE
)

def _keyword_score(text: str) -> float:
    lower = text.lower()
    yes_hits = len(re.findall(r'\byes\b|\bis ironic\b', lower))
    no_hits  = len(re.findall(r'\bno\b|\bis not ironic\b|\bnon.ironic\b', lower))
    return float(yes_hits - no_hits)


def parse_prediction(text: str):
    """
    Returns
    -------
    pred  : int   — 1 (ironic), 0 (non-ironic), -1 (uncertain)
    score : float — proxy confidence; N/A-style 0.0 for uncertain
    """
    text  = str(text).strip()
    first = text.split("\n")[0].strip()

    # ── Stage 0: Gemma hedging phrases → uncertain (BUG 3 FIX) ────────
    if HEDGING_PATTERNS.match(first):
        return -1, 0.0

    first_l = first.lower()

    # ── Stage 1: explicit yes/no opening (BUG 2 FIX: + 'potentially yes') ─
    if re.match(r'^\s*(potentially\s+)?yes[\s,.]', first_l):
        return 1, max(_keyword_score(text), 0.1)
    if re.match(r'^\s*no[\s,.]', first_l):
        return 0, min(_keyword_score(text), -0.1)

    # ── Stage 2: irony phrase in first line ─────────────────────────────
    if re.search(r'\bis ironic\b', first_l) and not re.search(r'not ironic', first_l):
        return 1, _keyword_score(text)
    if re.search(r'is not ironic|non.ironic', first_l):
        return 0, _keyword_score(text)

    # ── Stage 3: keyword balance over full text ──────────────────────────
    score = _keyword_score(text)
    if score > 0: return 1, score
    if score < 0: return 0, score

    return -1, 0.0  # uncertain / truncated


results     = df["model_output"].apply(parse_prediction)
df["pred"]  = [r[0] for r in results]
df["score"] = [r[1] for r in results]

n_uncertain = (df["pred"] == -1).sum()
print(f"Uncertain / truncated outputs : {n_uncertain} / {len(df)}")
print("\nPrediction distribution per condition:")
print(df.groupby(["condition", "pred"]).size().rename({1:'ironic',0:'non-ironic',-1:'uncertain'}).to_string())

# BUG 4 WARNING
c3_uncertain = (df[df["condition"]=="C3_prompting_style"]["pred"] == -1).sum()
c3_total     = (df["condition"]=="C3_prompting_style").sum()
if c3_uncertain == c3_total:
    print(f"\n⚠️  C3 WARNING: ALL {c3_total} outputs are truncated (no verdict extractable).")
    print("   Cause: max_new_tokens was set too low for chain-of-thought prompts.")
    print("   Fix:   Re-run Gemma inference on C3 with a larger max_new_tokens budget.")

## Step 4 — Condition-Level Metrics

In [ ]:
label_map = {1: "ironic", 0: "non-ironic", -1: "uncertain"}
condition_results = {}

log("[ZERO-SHOT RESULTS — CONDITION SUMMARY]")
log(f"{'Condition':<25}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}  "
    f"{'F1 ironic':>10}  {'F1 non-ironic':>14}  {'Unknowns':>9}")
log("-" * 93)

for cond_name, group in df.groupby("condition"):
    grp = group[group["pred"] != -1].copy()
    condition_results[cond_name] = group
    n_unk = (group["pred"] == -1).sum()

    if len(grp) == 0:
        log(f"{cond_name:<25}  {len(group):>4}  {'N/A':>9}  {'N/A':>9}  "
            f"{'N/A':>10}  {'N/A':>14}  {n_unk:>9}")
        continue

    labels       = grp["label"].tolist()
    preds        = grp["pred"].tolist()
    macro_f1     = f1_score(labels, preds, average="macro")
    acc          = accuracy_score(labels, preds)
    f1_ironic    = f1_score(labels, preds, pos_label=1, average="binary")
    f1_nonironic = f1_score(labels, preds, pos_label=0, average="binary")

    log(f"{cond_name:<25}  {len(group):>4}  {macro_f1:>9.4f}  {acc:>9.4f}  "
        f"{f1_ironic:>10.4f}  {f1_nonironic:>14.4f}  {n_unk:>9}")

log()

## Step 5 — Per-Level Breakdown

In [ ]:
log("[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]")
log(f"{'Condition':<25}  {'Level':<8}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}  {'Unknowns':>9}")
log("-" * 77)

for cond_name, group in condition_results.items():
    for level, grp in group.groupby("level"):
        valid = grp[grp["pred"] != -1]
        n_unk = (grp["pred"] == -1).sum()
        if len(valid) == 0:
            log(f"{cond_name:<25}  {level:<8}  {len(grp):>4}  {'N/A':>9}  {'N/A':>9}  {n_unk:>9}")
            continue
        lf1  = f1_score(valid["label"], valid["pred"], average="macro")
        lacc = accuracy_score(valid["label"], valid["pred"])
        log(f"{cond_name:<25}  {level:<8}  {len(grp):>4}  {lf1:>9.4f}  {lacc:>9.4f}  {n_unk:>9}")

log()

## Step 6 — Per-Item Predictions

In [ ]:
log("[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]")

for cond_name, group in condition_results.items():
    log(f"\n  Condition: {cond_name}")
    log(f"  {'item_id':<15}  {'level':<8}  {'true label':<12}  "
        f"{'predicted':<12}  {'score':>8}  {'correct'}")
    log("  " + "-" * 72)

    for _, row in group.iterrows():
        pred_label  = label_map[row["pred"]]
        correct_str = "N/A" if row["pred"] == -1 else ("YES" if row["label"] == row["pred"] else "NO")
        score_str   = "N/A" if row["pred"] == -1 else f"{row['score']:>+8.3f}"
        log(f"  {row['item_id']:<15}  {row['level']:<8}  "
            f"{label_map[row['label']]:<12}  "
            f"{pred_label:<12}  "
            f"{score_str:>8}  {correct_str}")

log()
log(f"Run finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nAll results saved to: {RESULTS_FILE}")